# Notebook 1: Base Stack Export

Exports multi-band rasters at MODIS scale in standard geographic coordinates (WGS84) per basin.
To completely bypass GEE's **User memory limit exceeded** errors, we split the 34-band composite into **three modular, lightweight assets** exported in parallel:

1. **`NppStack` (24 bands)**: MODIS NPP median + 23 annual bands. Already at MODIS scale (zero `reduceResolution` memory overhead).
2. **`GediStack` (3 bands)**: GEDI L2B UOI, N count, and L2A rh98 height (only 3 `reduceResolution` chains).
3. **`CovStack` (7 bands)**: Flood frequency, forest fraction, elevation, slope, hnd, precip, clay (only 7 `reduceResolution` chains).

### Running downstream:
Stage 2 loads these three static assets and concatenates them in one millisecond via `ee.Image.cat([npp, gedi, covs])`, running downstream calculations with **zero** live reprojection memory overhead!

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Output destination
GEE_PROJECT = 'quantum-bonus-434714-t2'
ASSET_ROOT = f'projects/{GEE_PROJECT}/assets/DefaunationFromSpace'

# Study Regions — per-basin exports avoid spanning the Atlantic
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Years
YEARS = list(range(2001, 2024))  # 2001-2023 inclusive

# GEE Dataset IDs
MODIS_NPP     = 'MODIS/061/MOD17A3HGF'
GLOFAS        = 'JRC/CEMS_GLOFAS/FloodHazard/v2_1'
MERIT_HYDRO   = 'MERIT/Hydro/v1_0_1'
FOREST_MASK   = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS  = 10   # Undisturbed since ~1982
SRTM          = 'USGS/SRTMGL1_003'
GEDI_L2B      = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
GEDI_L2A      = 'LARSE/GEDI/GEDI02_A_002_MONTHLY'
CHIRPS        = 'UCSB-CHG/CHIRPS/DAILY'
SOILGRIDS_CLAY = 'OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02'

# GEDI date range
GEDI_START = '2020-01-01'
GEDI_END   = '2023-12-31'

# CHIRPS date range (climatological mean)
PRECIP_START = '2001-01-01'
PRECIP_END   = '2023-12-31'

# --- MODIS NPP Reference Projection ---
_modis_col = ee.ImageCollection(MODIS_NPP).select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = MODIS_PROJ.nominalScale()

print("\u2713 Configuration loaded.")
print(f"  Basins: {[b[0] for b in BASINS]}")
print(f"  Asset root: {ASSET_ROOT}")

In [ ]:
# =============================================================================
# BLOCK 2: BAND-BUILDING FUNCTIONS
# =============================================================================

def build_npp_bands(basin_geom):
    modis = ee.ImageCollection(MODIS_NPP).select('Npp').filterBounds(basin_geom)
    
    def get_annual(year):
        year = ee.Number(year)
        return modis.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).first().clip(basin_geom).set('year', year)
    
    annual_imgs = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(get_annual)
    )
    
    median_npp = annual_imgs.median().clip(basin_geom).rename('Npp_median')
    annual_npp = annual_imgs.toBands().clip(basin_geom)
    band_names = [f'NPP_{y}' for y in YEARS]
    annual_npp = annual_npp.rename(band_names)
    
    return median_npp, annual_npp

def build_flood_frequency(basin_geom):
    glofas_col = ee.ImageCollection(GLOFAS).filterBounds(basin_geom)
    glofas = glofas_col.mosaic().clip(basin_geom)
    depth_bands = ['RP10_depth', 'RP20_depth', 'RP50_depth', 'RP75_depth',
                   'RP100_depth', 'RP200_depth', 'RP500_depth']
    
    flood_freq = glofas.select(depth_bands).gte(0).reduce(ee.Reducer.sum()).unmask()
    
    flood_proj = ee.Image(glofas_col.first()).projection()
    hnd_mask = ee.Image(MERIT_HYDRO).select('hnd').gt(0).clip(basin_geom)
    flood_freq = flood_freq.setDefaultProjection(crs=flood_proj).updateMask(hnd_mask)
    
    flood_reduced = flood_freq.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('flood_freq')
    
    return flood_reduced

def build_forest_fraction(basin_geom):
    tmf_col = ee.ImageCollection(FOREST_MASK).filterBounds(basin_geom)
    tmf_proj = tmf_col.first().projection()
    
    forest = tmf_col.mosaic().eq(FOREST_CLASS).clip(basin_geom).setDefaultProjection(tmf_proj)
    
    forest_frac = forest.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('forest_fraction')
    
    return forest_frac

def build_terrain(basin_geom):
    srtm = ee.Image(SRTM).clip(basin_geom)
    srtm_proj = srtm.select('elevation').projection()
    
    elev = srtm.select('elevation').setDefaultProjection(srtm_proj)
    elev_reduced = elev.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('elevation')
    
    slp = ee.Terrain.slope(srtm).clip(basin_geom).setDefaultProjection(srtm_proj)
    slp_reduced = slp.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('slope')
    
    return elev_reduced, slp_reduced

def build_hnd(basin_geom):
    hnd = ee.Image(MERIT_HYDRO).select('hnd').clip(basin_geom)
    hnd_proj = hnd.projection()
    
    hnd_reduced = hnd.setDefaultProjection(hnd_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).rename('hnd')
    
    return hnd_reduced

def build_gedi(basin_geom):
    gedi_b = ee.ImageCollection(GEDI_L2B).filterBounds(basin_geom).filterDate(GEDI_START, GEDI_END)
    native_b_proj = gedi_b.first().projection()
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        return uoi.clamp(0, 1).rename('UOI')
    
    uoi_col = gedi_b.map(calc_uoi)
    
    uoi_mean = uoi_col.mean().clip(basin_geom).rename('GEDI_UOI').setDefaultProjection(native_b_proj)
    uoi_reduced = uoi_mean.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    
    uoi_count = uoi_col.count().clip(basin_geom).rename('GEDI_N').setDefaultProjection(native_b_proj)
    n_reduced = uoi_count.reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535)
    
    gedi_a = ee.ImageCollection(GEDI_L2A).filterBounds(basin_geom).filterDate(GEDI_START, GEDI_END)
    native_a_proj = gedi_a.first().projection()
    
    rh98_mean = gedi_a.select('rh98').mean().clip(basin_geom).rename('GEDI_rh98').setDefaultProjection(native_a_proj)
    rh98_reduced = rh98_mean.reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535)
    
    return uoi_reduced, n_reduced, rh98_reduced

def build_precip(basin_geom):
    chirps = ee.ImageCollection(CHIRPS).filterBounds(basin_geom).filterDate(PRECIP_START, PRECIP_END)
    chirps_proj = chirps.first().projection()
    
    def annual_total(year):
        year = ee.Number(year)
        return chirps.filter(
            ee.Filter.calendarRange(year, year, 'year')
        ).sum().set('year', year)
    
    annual_precip = ee.ImageCollection.fromImages(
        ee.List(YEARS).map(annual_total)
    )
    mean_precip = annual_precip.mean().clip(basin_geom).rename('precip').setDefaultProjection(chirps_proj)
    
    return mean_precip

def build_clay(basin_geom):
    clay = ee.Image(SOILGRIDS_CLAY).clip(basin_geom)
    clay_mean = clay.reduce(ee.Reducer.mean()).rename('clay')
    clay_proj = clay.select(0).projection()
    
    clay_reduced = clay_mean.setDefaultProjection(clay_proj).reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    )
    
    return clay_reduced

print("\u2713 All band functions defined.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running modular unit tests (using Congo BBox)...\n")
    passed = 0
    total = 3
    
    try:
        print("  [1/3] Validating NPP Stack (24 bands)...")
        npp_med, npp_ann = build_npp_bands(CONGO_BBOX)
        npp_stack = ee.Image.cat([npp_med, npp_ann]).toFloat()
        assert len(npp_stack.bandNames().getInfo()) == 24
        passed += 1
        print("    \u2713 NPP Stack verified successfully!")
    except Exception as e:
        print(f"    \u2717 NPP Stack FAILED: {e}")
        
    try:
        print("  [2/3] Validating GEDI Stack (3 bands)...")
        uoi, n, rh98 = build_gedi(CONGO_BBOX)
        gedi_stack = ee.Image.cat([uoi, n, rh98]).toFloat()
        assert len(gedi_stack.bandNames().getInfo()) == 3
        passed += 1
        print("    \u2713 GEDI Stack verified successfully!")
    except Exception as e:
        print(f"    \u2717 GEDI Stack FAILED: {e}")
        
    try:
        print("  [3/3] Validating Covariates Stack (7 bands)...")
        flood = build_flood_frequency(CONGO_BBOX)
        forest = build_forest_fraction(CONGO_BBOX)
        elev, slope = build_terrain(CONGO_BBOX)
        hnd = build_hnd(CONGO_BBOX)
        precip = build_precip(CONGO_BBOX)
        clay = build_clay(CONGO_BBOX)
        cov_stack = ee.Image.cat([flood, forest, elev, slope, hnd, precip, clay]).toFloat()
        assert len(cov_stack.bandNames().getInfo()) == 7
        passed += 1
        print("    \u2713 Covariates Stack verified successfully!")
    except Exception as e:
        print(f"    \u2717 Covariates Stack FAILED: {e}")
        
    print(f"\n{'='*60}")
    if passed == total:
        print(f"  \u2713 ALL {passed}/{total} MODULAR STACKS VERIFIED")
    else:
        print(f"  \u2717 {passed}/{total} passed. Fix failures before proceeding.")
    print(f"{'='*60}")

run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: MODULAR PARALLEL EXPORTS
# =============================================================================

def safe_start(task, asset_id):
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_modular_stacks(dry_run=True):
    """Launches exports for 3 modular assets per basin (6 assets total) to GEE Assets.
    
    1. NPP Stack (24 bands - MODIS scale, 0 reduceResolution overhead)
    2. GEDI Stack (3 bands - heavy GEDI footprint reduction)
    3. Cov Stack (7 bands - heavy environmental grid reduction)
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        # --- NPP STACK ---
        npp_med, npp_ann = build_npp_bands(basin_geom)
        npp_stack = ee.Image.cat([npp_med, npp_ann]).toFloat()
        npp_id = f'{ASSET_ROOT}/NppStack_{basin_name}'
        
        task_npp = ee.batch.Export.image.toAsset(
            image=npp_stack,
            description=f'NppStack_{basin_name}',
            assetId=npp_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_npp, npp_id))
        
        # --- GEDI STACK ---
        uoi, n, rh98 = build_gedi(basin_geom)
        gedi_stack = ee.Image.cat([uoi, n, rh98]).toFloat()
        gedi_id = f'{ASSET_ROOT}/GediStack_{basin_name}'
        
        task_gedi = ee.batch.Export.image.toAsset(
            image=gedi_stack,
            description=f'GediStack_{basin_name}',
            assetId=gedi_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_gedi, gedi_id))
        
        # --- COVARIATES STACK ---
        flood = build_flood_frequency(basin_geom)
        forest = build_forest_fraction(basin_geom)
        elev, slope = build_terrain(basin_geom)
        hnd = build_hnd(basin_geom)
        precip = build_precip(basin_geom)
        clay = build_clay(basin_geom)
        
        cov_stack = ee.Image.cat([
            flood, forest, elev, slope, hnd, precip, clay
        ]).toFloat()
        cov_id = f'{ASSET_ROOT}/CovStack_{basin_name}'
        
        task_cov = ee.batch.Export.image.toAsset(
            image=cov_stack,
            description=f'CovStack_{basin_name}',
            assetId=cov_id,
            region=basin_geom,
            scale=MODIS_SCALE,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_cov, cov_id))
        
    print(f"\u2713 {len(tasks)} parallel modular tasks configured (3 per basin):")
    for _, aid in tasks:
        print(f"    {aid}")
        
    if dry_run:
        print("\nDRY RUN. Call export_modular_stacks(dry_run=False) to launch.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
            print(f"  \u2713 Started modular task: {asset_id.split('/')[-1]}")
        print("\n\u2713 All 6 parallel tasks started successfully!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

export_modular_stacks(dry_run=True)